In [7]:
import json
import requests
from src.data.make_dataset import build_interim_dataset, load_params
from src.features.build_features import build_features_dataset
from src.config import resolve_path
import pandas as pd

# Generando datos nuevos

In [3]:
newData = resolve_path('data/external')
datasetPath = resolve_path('data/external/interim')
featuresPath = resolve_path('data/external/processed')
paramsPath = resolve_path('params.yaml')
params = load_params(paramsPath)
build_interim_dataset(params,raw_dir=newData, out_dir=datasetPath)


Validación completada: dataset consistente.


PosixPath('/home/angelbri/MLopsProject/MLops_Equipo12/data/external/interim/mibici_data.parquet')

In [5]:
datasetPath = resolve_path('data/external/interim/mibici_data.parquet')
featuresPath = resolve_path('data/external/processed')
build_features_dataset(params,datasetPath,featuresPath)

/home/angelbri/MLopsProject/MLops_Equipo12/data/external/interim/mibici_data.parquet


PosixPath('/home/angelbri/MLopsProject/MLops_Equipo12/data/external/processed/mibici_features.parquet')

In [20]:
proccessedData = resolve_path('data/external/processed/mibici_features.parquet')
df = pd.read_parquet(proccessedData)
df[df.demanda > 10]

,Origen_Id,fecha_hora,demanda,lag_1h,lag_2h,lag_6h,lag_12h,lag_24h,lag_7d,media_24h,...,hora,dia_semana,mes,es_fin_semana,hour_sin,hour_cos,dow_sin,dow_cos,mes_sin,mes_cos
205,10,2026-10-16 13:00:00,14,6.0,8.0,3.0,0.0,6.0,3.0,2.458333,...,13,4,10,0,-0.258819,-9.659258e-01,-0.433884,-0.900969,-0.866025,0.5
320,10,2026-10-21 08:00:00,13,2.0,3.0,0.0,7.0,6.0,3.0,2.875000,...,8,2,10,0,0.866025,-5.000000e-01,0.974928,-0.222521,-0.866025,0.5
512,10,2026-10-29 08:00:00,11,2.0,0.0,0.0,3.0,4.0,5.0,1.791667,...,8,3,10,0,0.866025,-5.000000e-01,0.433884,-0.900969,-0.866025,0.5
1880,102,2026-10-14 08:00:00,13,8.0,2.0,0.0,2.0,5.0,9.0,2.625000,...,8,2,10,0,0.866025,-5.000000e-01,0.974928,-0.222521,-0.866025,0.5
1904,102,2026-10-15 08:00:00,11,7.0,4.0,0.0,5.0,13.0,6.0,2.625000,...,8,3,10,0,0.866025,-5.000000e-01,0.433884,-0.900969,-0.866025,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265098,87,2026-10-13 18:00:00,11,4.0,3.0,3.0,2.0,5.0,4.0,1.458333,...,18,1,10,0,-1.000000,-1.836970e-16,0.781831,0.623490,-0.866025,0.5
265834,88,2026-10-20 10:00:00,12,1.0,6.0,0.0,0.0,3.0,4.0,2.583333,...,10,1,10,0,0.500000,-8.660254e-01,0.781831,0.623490,-0.866025,0.5
266394,9,2026-10-19 18:00:00,12,2.0,4.0,3.0,3.0,2.0,0.0,2.416667,...,18,0,10,0,-1.000000,-1.836970e-16,0.000000,1.000000,-0.866025,0.5
266606,9,2026-10-28 14:00:00,11,2.0,3.0,3.0,0.0,5.0,7.0,2.291667,...,14,2,10,0,-0.500000,-8.660254e-01,0.974928,-0.222521,-0.866025,0.5


# Probando API

In [23]:
features = [
    "lag_1h",
    "lag_2h",
    "lag_6h",
    "lag_12h",
    "lag_24h",
    "lag_7d",
    "media_24h",
    "rolling_max_6h",
    "rolling_max_12h",
    "rolling_max_24h",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "mes_sin",
    "mes_cos",
    "Origen_Id",
    "es_fin_semana",
]

payload = df.loc[1880, features].to_dict()

payload

{'lag_1h': 8.0,
 'lag_2h': 2.0,
 'lag_6h': 0.0,
 'lag_12h': 2.0,
 'lag_24h': 5.0,
 'lag_7d': 9.0,
 'media_24h': 2.625,
 'rolling_max_6h': 8.0,
 'rolling_max_12h': 8.0,
 'rolling_max_24h': 8.0,
 'hour_sin': 0.8660254037844387,
 'hour_cos': -0.4999999999999998,
 'dow_sin': 0.9749279121818236,
 'dow_cos': -0.22252093395631434,
 'mes_sin': -0.8660254037844386,
 'mes_cos': 0.5000000000000001,
 'Origen_Id': '102',
 'es_fin_semana': 0}

In [24]:
r = requests.post(
    "http://localhost:8000/predict",
    json=payload,
)

print(r.status_code)
print(r.json())

200
{'predictions': [6.717385292053223]}
